### Teste de permutação — idade em EO (LogReg e SVM)

Passo de **evidência interna** após o screening exploratório em `07_ml_age_classification.ipynb` (não é validação confirmatória independente).

- **Tarefa:** só **EO** (escolhida após comparar EO/EC/Diff na mesma amostra)
- **Modelos:** LogReg_L2 e SVM_RBF (`class_weight="balanced"`)
- **Métrica principal:** `roc_auc`; **complementar:** `balanced_accuracy`
- **Null:** embaralha rótulos etários `y` (features fixas), `N_PERM=5000`
- **p-value bruto:** `(1 + #{null ≥ observed}) / (N + 1)`
- **Correção:** Bonferroni para 4 testes (2 modelos × 2 métricas), `α = 0,05/4 = 0,0125` — limite conservador *neste* bloco; a seleção prévia de tarefa/modelos no screening mantém o caráter exploratório

Espelho em script: `src/models/ml_classification.py` (`run_eo_permutation`).

**Features:** alinhadas por nome de eletrodo (mesmo dataset processado dos notebooks 07/09).


### Imports


In [15]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC


### Parâmetros e dados (EO)


In [ ]:
DATASET_PATH = Path("../../data/processed/dataset_features.parquet")
RANDOM_STATE = 42
N_SPLITS = 5
N_PERM = 5000  # execução final; 50 basta para smoke-test
CANDIDATE_MODELS = ("LogReg_L2", "SVM_RBF")
PERM_METRICS = ("roc_auc", "balanced_accuracy")  # principal, complementar
N_TESTS = 4  # 2 modelos × 2 métricas
ALPHA = 0.05
ALPHA_BONF = ALPHA / N_TESTS  # 0.0125

YOUNG_AGES = {"20-25", "25-30", "30-35", "35-40"}
OLDER_AGES = {"55-60", "60-65", "65-70", "70-75", "75-80"}

NODE_PREFIXES = (
    "degree_",
    "clustering_",
    "betweenness_",
    "local_efficiency_",
    "hub_frequency_",
    "weighted_degree_",
)
GLOBAL_FEATURES = [
    "global_efficiency",
    "edges_mean",
    "edges_std",
    "edges_cv",
]

df = pd.read_parquet(DATASET_PATH)
df = df[df["condition"].isin(["EO", "EC"])].copy()
df["age"] = df["age"].astype(str)

duplicates = df.duplicated(subset=["subject_id", "condition"], keep=False)
assert not duplicates.any(), (
    "Existem registros duplicados por participante e condição."
)


def map_age_group(age: str):
    if age in YOUNG_AGES:
        return "young"
    if age in OLDER_AGES:
        return "older"
    return None


df["age_group"] = df["age"].map(map_age_group)
df = df[df["age_group"].notna()].copy()
df["y"] = df["age_group"].map({"young": 0, "older": 1}).astype(int)

FEATURE_COLS = [
    c
    for c in df.columns
    if c.startswith(NODE_PREFIXES) or c in GLOBAL_FEATURES
]
FEATURE_COLS = (
    df[FEATURE_COLS]
    .select_dtypes(include="number")
    .dropna(axis=1, how="all")
    .columns.tolist()
)

eo = df[df["condition"] == "EO"].drop_duplicates("subject_id").copy()
X = eo[FEATURE_COLS].copy()
y = eo["y"].copy()
X.index = eo["subject_id"].values
y.index = eo["subject_id"].values

print(
    f"EO: n={len(y)}, features={X.shape[1]}, "
    f"y={y.value_counts().to_dict()}, N_PERM={N_PERM}"
)


### Modelos candidatos


In [ ]:
def get_candidate_models():
    return {
        "LogReg_L2": Pipeline(
            [
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
                (
                    "clf",
                    LogisticRegression(
                        max_iter=5000,
                        class_weight="balanced",
                        random_state=RANDOM_STATE,
                    ),
                ),
            ]
        ),
        "SVM_RBF": Pipeline(
            [
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
                (
                    "clf",
                    SVC(
                        kernel="rbf",
                        class_weight="balanced",
                        probability=True,
                        random_state=RANDOM_STATE,
                    ),
                ),
            ]
        ),
    }


models = get_candidate_models()
list(models)


### Funções de CV e permutação


In [18]:
def cv_mean_scores(model, X, y, metrics=PERM_METRICS):
    cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    scoring = {m: m for m in metrics}
    scores = cross_validate(model, X, y, cv=cv, scoring=scoring)
    return {m: float(np.mean(scores[f"test_{m}"])) for m in metrics}


def permutation_test(model, X, y, n_perm=N_PERM, metrics=PERM_METRICS, random_state=RANDOM_STATE):
    observed = cv_mean_scores(model, X, y, metrics=metrics)
    rng = np.random.default_rng(random_state)
    y_values = np.asarray(y)
    nulls = {m: np.empty(n_perm, dtype=float) for m in metrics}

    for i in range(n_perm):
        y_perm = pd.Series(rng.permutation(y_values), index=y.index)
        null_scores = cv_mean_scores(model, X, y_perm, metrics=metrics)
        for m in metrics:
            nulls[m][i] = null_scores[m]

    p_values = {}
    for m in metrics:
        obs = observed[m]
        p_values[m] = float((1 + int(np.sum(nulls[m] >= obs))) / (n_perm + 1))
    return {"observed": observed, "nulls": nulls, "p_values": p_values}


### Rodar permutação

Pode levar bastante tempo (`N_PERM=5000 × 2 modelos × 5 folds`). Para smoke-test, use temporariamente `N_PERM = 50`.


In [ ]:
rows = []
null_store = {}

for name in CANDIDATE_MODELS:
    print(f"=== EO | {name} | N_PERM={N_PERM} ===")
    out = permutation_test(models[name], X, y, n_perm=N_PERM)
    null_store[name] = out["nulls"]
    for metric in PERM_METRICS:
        rows.append(
            {
                "model": name,
                "metric": metric,
                "observed": out["observed"][metric],
                "p": out["p_values"][metric],
                "n_perm": N_PERM,
            }
        )
        print(
            f"  {metric}: observed={out['observed'][metric]:.3f} | "
            f"p={out['p_values'][metric]:.4f}"
        )

df_perm = pd.DataFrame(rows)
df_perm["alpha_bonf"] = ALPHA_BONF
df_perm["sig_bonferroni"] = df_perm["p"] < ALPHA_BONF
df_perm


### Tabela resumo


In [ ]:
print(f"α Bonferroni (4 testes) = {ALPHA_BONF:.4f}")
summary = df_perm.pivot(
    index="model", columns="metric", values=["observed", "p", "sig_bonferroni"]
)
summary


### Histogramas da distribuição nula


In [ ]:
fig = make_subplots(
    rows=len(CANDIDATE_MODELS),
    cols=len(PERM_METRICS),
    subplot_titles=[
        f"{m} | {metric}" for m in CANDIDATE_MODELS for metric in PERM_METRICS
    ],
    vertical_spacing=0.12,
    horizontal_spacing=0.08,
)

for i, name in enumerate(CANDIDATE_MODELS):
    for j, metric in enumerate(PERM_METRICS):
        null = null_store[name][metric]
        obs = float(
            df_perm.loc[
                (df_perm["model"] == name) & (df_perm["metric"] == metric),
                "observed",
            ].iloc[0]
        )
        p = float(
            df_perm.loc[
                (df_perm["model"] == name) & (df_perm["metric"] == metric),
                "p",
            ].iloc[0]
        )
        fig.add_trace(
            go.Histogram(x=null, nbinsx=30, name=f"{name}-{metric}", showlegend=False),
            row=i + 1,
            col=j + 1,
        )
        fig.add_vline(
            x=obs,
            line_dash="dash",
            line_color="crimson",
            annotation_text=f"obs={obs:.3f} p={p:.3f}",
            annotation_position="bottom right",
            row=i + 1,
            col=j + 1,
        )

fig.update_layout(
    height=700,
    title_text="Null de rótulos embaralhados (EO) vs score observado",
    template="plotly_white",
    hovermode=False,
)
fig.update_xaxes(range=[0, 1])
fig.show(config={"displayModeBar": False, "staticPlot": False})


### Interpretação

- **Métrica principal:** `roc_auc`; `balanced_accuracy` é complementar.
- Sob H0 (rótulos etários independentes das features), o score observado deveria cair no miolo da distribuição nula.
- Reportar **p bruto** e se `p < α_Bonferroni` (`0,0125` para 4 testes neste bloco).
- Bonferroni é limiar **conservador**; não se afirma aqui um “número efetivo” de testes sem estimá-lo formalmente.
- Como EO e os modelos foram escolhidos **após** o screening na **mesma amostra**, os p são pós-seleção: evidência **exploratória/preliminar**, não confirmação independente nem prova de generalização populacional.
- n = 54 e p ≫ n reforçam o caráter exploratório; o sinal (se mantido com `N_PERM=5000`) merece validação em amostra independente ou maior.

#### Redação sugerida para o TCC (atualizar números após `N_PERM=5000`)

No teste de permutação com *N* embaralhamentos, o SVM com kernel RBF apresentou ROC-AUC média de *X* e p=*Y* na classificação das faixas etárias com as redes da condição de olhos abertos. Compare *Y* ao limite de Bonferroni para quatro comparações (α=0,0125). Entretanto, como os modelos e a condição foram selecionados após uma etapa exploratória envolvendo diferentes tarefas, o resultado deve ser interpretado como evidência preliminar — há um sinal preditivo associado à idade que merece validação em amostra independente — e não como validação confirmatória definitiva.

*(Com N_PERM=500 prévio: SVM ROC-AUC ≈ 0,769, p ≈ 0,006 — abaixo de 0,0125; recalcular após a execução final.)*

Anteriores: `07_ml_age_classification.ipynb`.

Próximo passo (robustez / interpretabilidade): `09_ml_age_robustness_interpretability.ipynb`.
